# MoE Paper Figures

Produces exactly three figures for the paper:
1. Expert load balance (sanity check)
2. Kapanji group × Expert gate weight heatmap
3. Morphology heatmap (PWD users only, skips users with `gesture_features=None`)

**NOTE on 'all users' change:** If you want to switch from val+test subjects only to
ALL users (including train), you must rerun Cell 5 (the slow routing collection loop)
with a dataloader built over all PIDs. There is no way around this — the routing vectors
are only computed during the forward pass, so they must be re-collected for train users.
Change `VAL_PIDS` to `ALL_PIDS = set(TRAIN_PIDS) | VAL_PIDS` and rebuild the dataloader
before running cell 5.

## 0. Imports

In [1]:
import sys
import os
import json
import pickle
import warnings
from pathlib import Path

import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore', category=UserWarning)

PROJECT_ROOT = r"C:\Users\kdmen\Repos\pers-gest-cls\system"
sys.path.insert(0, PROJECT_ROOT)

from MOE.MOE_encoder import build_MOE_model
from MOE.MOE_analysis import RoutingCollector, RoutingAnalyzer, RoutingRecord
from MAML.mamlpp import mamlpp_adapt, _normalize_step_item, named_param_dict, FunctionalModel
from MAML.maml_data_pipeline import get_maml_dataloaders

print('Imports OK')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

Imports OK
PyTorch: 2.7.1+cu128
CUDA available: False


## 1. Load train/val PID split

In [2]:
SPLIT_JSON_PATH = r"C:\Users\kdmen\Repos\pers-gest-cls\system\fixed_user_splits\hpo_strat_kapanji_split.json"
FOLD_KEY = 0

KAPANJI_GROUPS = {
    'Non-disabled Kapanji':  {'P011', 'P010', 'P008', 'P006', 'P005', 'P004'},
    'Disabled Kapanji':      {'P111', 'P119', 'P124', 'P110', 'P112', 'P123',
                              'P132', 'P126', 'P131', 'P104', 'P116'},
    'Disabled Non-Kapanji':  {'P102', 'P114', 'P107', 'P103', 'P125', 'P127',
                              'P128', 'P118', 'P108', 'P122', 'P106', 'P115',
                              'P105', 'P109', 'P121'},
}

with open(SPLIT_JSON_PATH, 'r') as f:
    split_data = json.load(f)

fold        = split_data[FOLD_KEY]
TRAIN_PIDS  = fold['train']
HP_VAL_PIDS = set(fold['val'])
TEST_PIDS   = set(fold['test'])
VAL_PIDS    = HP_VAL_PIDS | TEST_PIDS
ALL_PIDS = set(TRAIN_PIDS) | VAL_PIDS

assert len(set(TRAIN_PIDS) & VAL_PIDS) == 0, 'PID leakage between train and val+test!'

print(f'Train PIDs ({len(TRAIN_PIDS)}): {sorted(TRAIN_PIDS)}')
print(f'Val+Test (used for routing analysis) ({len(VAL_PIDS)}): {sorted(VAL_PIDS)}')

Train PIDs (24): ['P006', 'P008', 'P010', 'P011', 'P102', 'P103', 'P106', 'P107', 'P108', 'P110', 'P111', 'P112', 'P114', 'P115', 'P118', 'P119', 'P122', 'P123', 'P124', 'P125', 'P126', 'P127', 'P128', 'P132']
Val+Test (used for routing analysis) (8): ['P004', 'P005', 'P104', 'P105', 'P109', 'P116', 'P121', 'P131']


## 2. Config

In [3]:
config = {
    # NOTE: OVERWROTE so that we run all PIDs, not just the withheld users!
    'train_PIDs':                          [],
    'val_PIDs':                            ALL_PIDS,

    'model_type':                          'DeepCNNLSTM',
    'use_MOE':                             True,
    'MOE_placement':                       'encoder',
    'num_experts':                         24,
    'MOE_ctx_hidden_dim':                  32,
    'MOE_ctx_out_dim':                     64,
    'MOE_gate_temperature':                1.1879664247660187,
    'MOE_top_k':                           7,
    'top_k':                               7,
    'MOE_dropout':                         0.022501513050004283,
    'MOE_expert_expand':                   1.0,
    'MOE_mlp_hidden_mult':                 1.0,
    'MOE_aux_coeff':                       0.08672942143224953,
    'apply_MOE_aux_loss_inner_outer':      'outer',
    'FILM_on_context_or_demo':             'context',
    'gate_type':                           'context_feature_demo',
    'expert_architecture':                 'MLP',
    'front_end_stride':                    0,
    'MOE_log_every':                       5,
    'MOE_plot_dir':                        None,
    'emg_in_ch':                           16,
    'imu_in_ch':                           72,
    'use_imu':                             True,
    'multimodal':                          True,
    'use_demographics':                    False,
    'use_film_x_demo':                     False,
    'demo_in_dim':                         12,
    'sequence_length':                     64,
    'augment':                             False,
    'feature_engr':                        None,
    'padding':                             0,
    'cnn_base_filters':                    64,
    'cnn_layers':                          3,
    'cnn_kernel':                          5,
    'cnn_filters':                         64,
    'groupnorm_num_groups':                8,
    'lstm_hidden':                         64,
    'lstm_layers':                         3,
    'dropout':                             0.1,
    'bidirectional':                       True,
    'head_type':                           'mlp',
    'use_GlobalAvgPooling':                True,
    'use_batch_norm':                      False,
    'meta_learning':                       True,
    'n_way':                               3,
    'k_shot':                              1,
    'q_query':                             9,
    'num_classes':                         10,
    'pretrain_num_classes':                10,
    'available_gesture_classes':           [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    'maml_inner_steps':                    10,
    'maml_inner_steps_eval':               10,
    'maml_alpha_init':                     0.0009888781900907544,
    'maml_alpha_init_eval':                0.006717556958813548,
    'maml_use_lslr':                       True,
    'use_lslr_at_eval':                    False,
    'maml_opt_order':                      'first',
    'maml_first_order_to_second_order_epoch': 1000000,
    'use_maml_msl':                        'hybrid',
    'maml_msl_num_epochs':                 39,
    'meta_batchsize':                      24,
    'label_smooth':                        0.05,
    'ft_label_smooth':                     0.0,
    'enable_inner_loop_optimizable_bn_params': False,
    'track_gradient_alignment':            False,
    'optimizer':                           'adam',
    'learning_rate':                       0.00010093304999603776,
    'weight_decay':                        0.0008325426470137959,
    'gradient_clip_max_norm':              10.0,
    'lr_scheduler_factor':                 0.1,
    'lr_scheduler_patience':              6,
    'use_cosine_outer_lr':                 False,
    'use_earlystopping':                   True,
    'earlystopping_min_delta':             0.005,
    'earlystopping_patience':              8,
    'pretrain_approach':                   None,
    'pretrained_model_filename':           None,
    'pretrain_dir':                        'C:\\Users\\kdmen\\Repos\\pers-gest-cls\\hpo_best_models\\',
    'maml_gesture_classes':                [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    'target_trial_reps':                   None,
    'train_reps':                          [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'val_reps':                            [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'episodes_per_epoch_train':            250,
    'num_eval_episodes':                   200,
    'num_workers':                         0,
    'use_label_shuf_meta_aug':             False,
    'seed':                                42,
    'debug_one_episode':                   False,
    'debug_five_episodes':                 False,
    'debug_one_user_only':                 False,
    'debug_verbose':                       False,
    'subject_specific_model':              False,
    'NOTS':                                True,
    'ablation_id':                         'M0',
    'dfs_load_path':                       'C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\meta-learning-sup-que-ds\\',
    'user_split_json_filepath':            SPLIT_JSON_PATH,
    'models_save_dir':                     None,
    'results_save_dir':                    r'C:\Users\kdmen\Repos\pers-gest-cls\paper_figures',
    'device':                              'cuda' if torch.cuda.is_available() else 'cpu',
    'demo_dim_labels': [
        'time_disabled', 'age', 'BMI', 'DASH_score',
        'disability_coding_MD', 'disability_coding_No_Disability',
        'disability_coding_PN', 'disability_coding_SCI', 'disability_coding_other',
        'handedness_Right',
        'gender_Non-binary', 'gender_Woman',
    ],
}

print(f"Device          : {config['device']}")
print(f"Val PIDs ({len(config['val_PIDs'])}): {sorted(config['val_PIDs'])}")

Device          : cpu
Val PIDs (32): ['P004', 'P005', 'P006', 'P008', 'P010', 'P011', 'P102', 'P103', 'P104', 'P105', 'P106', 'P107', 'P108', 'P109', 'P110', 'P111', 'P112', 'P114', 'P115', 'P116', 'P118', 'P119', 'P121', 'P122', 'P123', 'P124', 'P125', 'P126', 'P127', 'P128', 'P131', 'P132']


## 3. Load data (val+test users)

In [4]:
TENSOR_DICT_PATH = r"C:\Users\kdmen\Repos\pers-gest-cls\dataset\meta-learning-sup-que-ds\segfilt_withMorphology_tensor_dict.pkl"

_, val_dl = get_maml_dataloaders(config, TENSOR_DICT_PATH)
print(f'Val dataloader: {len(val_dl)} episodes')

[get_maml_dataloaders] Train PIDs (0): []
[get_maml_dataloaders] Val   PIDs (32):   ['P004', 'P005', 'P006', 'P008', 'P010', 'P011', 'P102', 'P103', 'P104', 'P105', 'P106', 'P107', 'P108', 'P109', 'P110', 'P111', 'P112', 'P114', 'P115', 'P116', 'P118', 'P119', 'P121', 'P122', 'P123', 'P124', 'P125', 'P126', 'P127', 'P128', 'P131', 'P132']
[get_maml_dataloaders] num_eval_episodes per val user: 200
[get_maml_dataloaders] Total val episodes in cache: 6400
[get_maml_dataloaders] use_label_shuf_meta_aug: False
Val dataloader: 6400 episodes


## 4. Load model checkpoint

In [5]:
def load_maml_checkpoint(checkpoint_path: Path) -> tuple:
    print(f"Loading MAML checkpoint: {checkpoint_path}")
    assert checkpoint_path.exists(), f"Checkpoint not found: {checkpoint_path}"

    ckpt   = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    config = ckpt["config"]

    device           = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    config["device"] = device

    if config["use_MOE"]:
        from MOE.MOE_encoder import build_MOE_model
        model = build_MOE_model(config)
    else:
        from pretraining.pretrain_models import build_model
        model = build_model(config)

    if config.get("maml_use_lslr", False):
        from MAML.mamlpp import PerParamPerStepLSLR, named_param_dict
        temp_params = named_param_dict(model, require_grad_only=True)
        model._lslr = PerParamPerStepLSLR(
            named_params = temp_params.items(),
            inner_steps  = config["maml_inner_steps"],
            init_lr      = config["maml_alpha_init"],
            learnable    = True,
            device       = device,
        ).to(device)

    raw_sd     = ckpt["model_state_dict"]
    remapped_sd = {}
    n_remapped  = 0
    for k, v in raw_sd.items():
        new_k = k.replace("ctx_proj.", "router.projector.").replace("ctx_proj-", "router-projector-")
        if new_k != k:
            n_remapped += 1
        remapped_sd[new_k] = v
    if n_remapped > 0:
        print(f"  [ckpt remap] Remapped {n_remapped} keys: ctx_proj -> router.projector")

    model.load_state_dict(remapped_sd)
    model.to(device)
    model.eval()
    print(f"  Best val acc    : {ckpt.get('best_val_acc', 'N/A')}")
    return model, config

In [6]:
del config

CHECKPOINT_PATH = r"C:\Users\kdmen\Repos\pers-gest-cls\models\final_eval_models\best_M0_model.pt"

model, loaded_config = load_maml_checkpoint(Path(CHECKPOINT_PATH))
config = loaded_config

# Restore fields not saved in checkpoint
config['demo_dim_labels'] = [
    'time_disabled', 'age', 'BMI', 'DASH_score',
    'disability_coding_MD', 'disability_coding_No_Disability',
    'disability_coding_PN', 'disability_coding_SCI', 'disability_coding_other',
    'handedness_Right', 'gender_Non-binary', 'gender_Woman',
]
config['num_eval_episodes'] = 200
config['val_PIDs']  = VAL_PIDS
config['train_PIDs'] = TRAIN_PIDS
config['user_split_json_filepath'] = SPLIT_JSON_PATH
config['results_save_dir'] = r'C:\Users\kdmen\Repos\pers-gest-cls\paper_figures'

E    = config['num_experts']
TOPK = config['MOE_top_k']
n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {config['model_type']} (MoE encoder) | {n_params:,} parameters")
print(f"E={E}, top_k={TOPK}, ideal selection freq per expert = {TOPK/E:.3f}")

Loading MAML checkpoint: C:\Users\kdmen\Repos\pers-gest-cls\models\final_eval_models\best_M0_model.pt
  Best val acc    : 0.9066927083333334
Model loaded: DeepCNNLSTM (MoE encoder) | 5,538,216 parameters
E=22, top_k=9, ideal selection freq per expert = 0.409


## 5. Collect routing data (post-adaptation, query-only)

**This is the slow cell.** If you saved a RoutingRecord previously, skip to Cell 5b to reload it.

**To switch to ALL users (including train):** Before running this cell, rebuild `val_dl` over
all PIDs by setting `config['val_PIDs'] = set(TRAIN_PIDS) | VAL_PIDS` and re-running Cell 3.
There is no shortcut — routing vectors for train users have never been computed.

In [ ]:
collector = RoutingCollector(
    num_experts = E,
    model_name  = config['model_type'],
)

device     = config['device']
multimodal = bool(config.get('multimodal', True))
n_episodes = 0
n_skipped  = 0

model.eval()

for step_item in val_dl:
    episodes = _normalize_step_item(step_item)
    for episode in episodes:
        support_batch = episode['support']
        query_batch   = episode['query']

        user_id = episode['user_id']
        if isinstance(user_id, (list, tuple)):
            episode_pids = [str(p) for p in user_id]
        else:
            episode_pids = str(user_id)

        theta_prime = mamlpp_adapt(model, config, support_batch, use_lslr_at_eval=False)
        f_q = FunctionalModel(model, theta_prime)

        qemg    = query_batch['emg'].to(device)
        qlabels = query_batch['labels'].to(device)
        qimu    = query_batch.get('imu')
        if qimu is not None and multimodal:
            qimu = qimu.to(device)
        else:
            qimu = None

        if isinstance(episode_pids, str):
            episode_pids = [episode_pids] * qemg.size(0)

        with torch.no_grad():
            out = f_q(qemg, qimu, return_routing=True)

        if not (isinstance(out, tuple) and len(out) == 2 and isinstance(out[1], dict)):
            n_skipped += 1
            continue

        _, routing_info = out
        gate_w = routing_info.get('gate_weights')
        if gate_w is None:
            n_skipped += 1
            continue

        demo         = query_batch.get('demo')
        gesture_ids  = query_batch.get('global_labels', qlabels).cpu()

        collector.add(
            gate_weights   = gate_w.cpu(),
            gesture_labels = gesture_ids,
            pids           = episode_pids,
            demographics   = demo.cpu() if demo is not None else None,
        )
        n_episodes += 1

    model.eval()

print(f'Collected {n_episodes} episodes, skipped {n_skipped}')
record = collector.finalize()
print(f'Total query samples : {record.gate_weights.shape[0]}')
print(f'Gate weights shape  : {record.gate_weights.shape}  (N x E)')
print(f'Unique users        : {len(set(record.pids))}')

## 5b. Save / reload RoutingRecord (skip Cell 5 on future runs)

In [ ]:
from MOE.MOE_analysis import save_routing_record, load_routing_record

RECORD_SAVE_PATH = r"C:\Users\kdmen\Repos\pers-gest-cls\paper_figures\routing_record_M0.pt"

# Uncomment to save after first run:
# save_routing_record(record, RECORD_SAVE_PATH)
# print(f'Saved to {RECORD_SAVE_PATH}')

# Uncomment to reload and skip cells 3–5 on future runs:
# record = load_routing_record(RECORD_SAVE_PATH)
# E      = record.gate_weights.shape[1]
# print(f'Loaded {record.gate_weights.shape[0]} samples, {len(set(record.pids))} users')

## 5c. Build user-level aggregates

In [ ]:
pids_arr   = np.array(record.pids)
gw_all     = record.gate_weights.float()

unique_pids = sorted(set(record.pids))
N_users     = len(unique_pids)

# User-level mean gate weight vectors — shape (N_users, E)
user_gw = np.stack([
    gw_all[pids_arr == uid].mean(0).numpy()
    for uid in unique_pids
])

print(f'N_users   : {N_users}')
print(f'user_gw shape: {user_gw.shape}  (N_users x E)')

---
# FIGURE 1 — Expert Load Balance

In [ ]:
analyzer = RoutingAnalyzer(record)
lb       = analyzer.load_balance()

fig, ax = plt.subplots(figsize=(max(8, E * 0.6 + 2), 4))
x = np.arange(E)
ax.bar(x - 0.2, lb['expert_soft_fraction'], 0.35, label='Mean soft weight',     color='#4C72B0')
ax.bar(x + 0.2, lb['expert_hard_fraction'], 0.35, label='Dominant expert freq', color='#DD8452')
ax.axhline(lb['ideal_fraction'], color='gray', linestyle='--',
           label=f"Ideal (1/E = {lb['ideal_fraction']:.2f})")
ax.set_xlabel('Expert index')
ax.set_ylabel('Fraction')
ax.set_title(
    f"Expert Load Balance | E={E}, top_k={TOPK}\n"
    f"N_samples={record.gate_weights.shape[0]}, N_users={N_users}"
)
ax.set_xticks(x)
ax.set_xticklabels([f'E{i}' for i in range(E)], rotation=90, fontsize=7)
ax.legend()
fig.tight_layout()
plt.show()

print(f"Hard imbalance ratio (max/min dom freq): {lb['hard_imbalance_ratio']:.2f}x")
print(f"Soft imbalance ratio (max/min mean wgt): {lb['soft_imbalance_ratio']:.2f}x")

---
# FIGURE 2 — Kapanji Group × Expert Gate Weight Heatmap

In [ ]:
all_kapanji_pids   = set().union(*KAPANJI_GROUPS.values())
unassigned         = set(unique_pids) - all_kapanji_pids
if unassigned:
    print(f"WARNING: {len(unassigned)} val PIDs have no Kapanji group: {sorted(unassigned)}")

global_mean_kap = user_gw.mean(axis=0)   # (E,)
group_names     = list(KAPANJI_GROUPS.keys())
n_groups        = len(group_names)
mat_kap         = np.zeros((n_groups, E))
group_sizes     = {}

for g_idx, group_name in enumerate(group_names):
    group_pid_set = KAPANJI_GROUPS[group_name]
    mask          = np.array([p in group_pid_set for p in unique_pids])
    n_in_group    = mask.sum()
    group_sizes[group_name] = n_in_group
    if n_in_group == 0:
        print(f"  WARNING: '{group_name}' has 0 users in val set. Row will be zero.")
        continue
    mat_kap[g_idx] = user_gw[mask].mean(axis=0) - global_mean_kap

print("Kapanji group user counts:")
for name, n in group_sizes.items():
    print(f"  {name}: {n} users")

vlim_kap = max(abs(mat_kap).max(), 0.005)

fig, ax = plt.subplots(figsize=(max(10, E * 0.55 + 2), max(3, n_groups * 0.7 + 1.5)))
im = ax.imshow(mat_kap, aspect='auto', vmin=-vlim_kap, vmax=vlim_kap, cmap='RdBu_r')
ax.set_xticks(range(E))
ax.set_xticklabels([f'E{i}' for i in range(E)], rotation=90, fontsize=6)
ax.set_yticks(range(n_groups))
ax.set_yticklabels([f"{n}  (n={group_sizes[n]} users)" for n in group_names], fontsize=9)
ax.set_xlabel('Expert')
ax.set_title(
    f"Kapanji Group × Expert — Gate Weight Contrast (group mean − global mean)\n"
    f"User-level | N_users={N_users} | top_k={TOPK}/{E}",
    fontsize=9,
)
plt.colorbar(im, ax=ax, label='Gate weight contrast\n(group mean − global mean)', shrink=0.8)
fig.tight_layout()
plt.show()

---
# FIGURE 3 — Morphology Heatmap (PWD users only)

Loads morphology data (`gesture_features`) directly from the tensor dict.
For each user, averages their gesture-level feature vectors into a single user-level vector.
Users with `gesture_features=None` (PWoD users and P124) are skipped and reported.

Rows = users with disabilities who have morphology data.
Columns = morphology feature dimensions (time_sec + OHE dummies: hand_used, midair_or_table,
muscles_1, muscles_union, lifted_arms, dynamic_gesture).

In [ ]:
# ── Load tensor dict ──────────────────────────────────────────────────────
with open(TENSOR_DICT_PATH, 'rb') as f:
    full_dict = pickle.load(f)

data_dict           = full_dict['data']
gesture_feature_cols = full_dict['gesture_feature_cols']   # list of column name strings
pwod_pids_from_dict  = full_dict['pwod_pids']              # set of PWoD PIDs (gesture_features=None)

print(f"Tensor dict loaded. Total PIDs in dict: {len(data_dict)}")
print(f"Gesture feature columns ({len(gesture_feature_cols)}): {gesture_feature_cols}")
print(f"PWoD PIDs (will be skipped): {sorted(pwod_pids_from_dict)}")

# ── Build user-level morphology matrix ────────────────────────────────────
# We only include users that are in the current routing analysis (unique_pids).
# For each included user, average gesture_features tensors across all gesture classes.
# Users with gesture_features=None are skipped (PWoD + P124).

morph_pids   = []   # PIDs included in the heatmap
morph_vecs   = []   # one averaged feature vector per included user
skipped_pids = []   # PIDs skipped because gesture_features is None

for pid in unique_pids:   # unique_pids from routing analysis
    if pid not in data_dict:
        # This would be a real data integrity issue, so raise rather than silently skip
        raise KeyError(f"PID {pid} is in routing analysis but not found in tensor dict!")

    pid_entries = data_dict[pid]   # dict: enc_gesture_id -> entry

    # Collect gesture_features from all gesture classes for this user
    feat_tensors = []
    for enc_gid, entry in pid_entries.items():
        gf = entry['gesture_features']
        if gf is not None:
            feat_tensors.append(gf)

    if len(feat_tensors) == 0:
        # All gestures returned None -> this user has no morphology data
        skipped_pids.append(pid)
        continue

    # Average across gesture classes to get one user-level morphology vector
    user_morph = torch.stack(feat_tensors).mean(dim=0).numpy()   # (D_gesture_features,)
    morph_pids.append(pid)
    morph_vecs.append(user_morph)

print(f"\nUsers included in morphology heatmap : {len(morph_pids)}")
print(f"Users skipped (gesture_features=None): {len(skipped_pids)}")
print(f"  Skipped PIDs: {sorted(skipped_pids)}")

assert len(morph_vecs) > 0, "No users with morphology data found — check tensor dict."

morph_matrix = np.stack(morph_vecs)   # (N_morph_users, D_gesture_features)
print(f"\nMorphology matrix shape: {morph_matrix.shape}  (users x features)")

In [ ]:
# ── Kapanji group labels for morphology users (for row annotation) ─────────
pid_to_kapanji = {}
for group_name, group_pid_set in KAPANJI_GROUPS.items():
    for pid in group_pid_set:
        pid_to_kapanji[pid] = group_name

# Sort rows by Kapanji group so the heatmap is organized
group_order = list(KAPANJI_GROUPS.keys())
morph_pids_sorted_idx = sorted(
    range(len(morph_pids)),
    key=lambda i: (
        group_order.index(pid_to_kapanji[morph_pids[i]])
        if morph_pids[i] in pid_to_kapanji else len(group_order),
        morph_pids[i]
    )
)

morph_pids_sorted   = [morph_pids[i] for i in morph_pids_sorted_idx]
morph_matrix_sorted = morph_matrix[morph_pids_sorted_idx]

# ── Row labels: PID + Kapanji group abbreviation ──────────────────────────
group_abbrevs = {
    'Non-disabled Kapanji': 'NDK',
    'Disabled Kapanji':     'DK',
    'Disabled Non-Kapanji': 'DNK',
}
row_labels = [
    f"{pid} [{group_abbrevs.get(pid_to_kapanji.get(pid, ''), '?')}]"
    for pid in morph_pids_sorted
]

# ── Column labels: shorten OHE dummy names for readability ────────────────
# Strip common prefixes to keep the heatmap readable
def shorten_col(col: str) -> str:
    replacements = [
        ('muscles_union_', 'mu_'),
        ('muscles_1_', 'm1_'),
        ('hand_used_', 'hand_'),
        ('midair_or_table_', 'mid_'),
    ]
    for old, new in replacements:
        col = col.replace(old, new)
    return col

col_labels = [shorten_col(c) for c in gesture_feature_cols]

# ── Plot ──────────────────────────────────────────────────────────────────
n_rows = len(morph_pids_sorted)
n_cols = morph_matrix_sorted.shape[1]

fig_h = max(5, n_rows * 0.45 + 2)
fig_w = max(10, n_cols * 0.55 + 3)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

im = ax.imshow(morph_matrix_sorted, aspect='auto', cmap='YlOrRd', interpolation='nearest')

ax.set_xticks(range(n_cols))
ax.set_xticklabels(col_labels, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(n_rows))
ax.set_yticklabels(row_labels, fontsize=8)
ax.set_xlabel('Morphology feature', fontsize=10)
ax.set_ylabel('Participant (sorted by Kapanji group)', fontsize=10)
ax.set_title(
    f"Morphology Features — PWD Users Only\n"
    f"N={n_rows} users | Averaged across gesture classes\n"
    f"[NDK=Non-disabled Kapanji, DK=Disabled Kapanji, DNK=Disabled Non-Kapanji]\n"
    f"Skipped {len(skipped_pids)} users with no morphology data: {sorted(skipped_pids)}",
    fontsize=9,
)
plt.colorbar(im, ax=ax, label='Feature value', shrink=0.8)

# Draw horizontal lines to separate Kapanji groups
current_group = None
for row_i, pid in enumerate(morph_pids_sorted):
    grp = pid_to_kapanji.get(pid, 'Unknown')
    if current_group is not None and grp != current_group:
        ax.axhline(row_i - 0.5, color='black', linewidth=1.5)
    current_group = grp

fig.tight_layout()
plt.show()

print("\nUsers included per group:")
for grp in group_order:
    pids_in_grp = [p for p in morph_pids_sorted if pid_to_kapanji.get(p) == grp]
    print(f"  {grp}: {pids_in_grp}")
print(f"\nSkipped (gesture_features=None): {sorted(skipped_pids)}")